# `ratio_study` - 01 Basics

### Ratio studies for Computer Assisted Mass Appraisal (CAMA), in Python

*Developed by **ASMA-DS**, the Data Science team within the Assessment Standards and Mass Appraisal department at the Municipal Property Assessment Corporation (MPAC), Ontario, Canada.*

---

This is the first of five notebooks. It covers what ratio studies are (section 1) and the basic workflow (section 2) on the anonymised sample dataset that ships inside the package. Run the cells top to bottom.

Everything here is aligned to the IAAO **Standard on Ratio Studies**. Where the package goes beyond the Standard, the notebook says so.

| Notebook | Covers |
|---|---|
| [01 Basics](01_basics.ipynb) | Sections 1-2: indicators, confidence intervals, compliance checks, IQR trimming, charts, comparisons |
| [02 Spatial](02_spatial.ipynb) | Section 3: ratio maps, Moran's I, hot/cold spots, the VEM map |
| [03 Reporting](03_reporting.ipynb) | Section 4: PDF / XLSX / CSV / PNG output |
| [04 Advanced diagnostics](04_advanced_diagnostics.ipynb) | Sections 5 and 7: ML / SHAP inequity diagnostics and sales-chasing detection |
| [05 Vertical equity](05_vertical_equity.ipynb) | Section 6: VEI, the Vertical Equity Panel, VEM, the Gini family |

All five use the anonymised sample dataset bundled with the package - no data preparation required.
Section numbers are kept from the original single-notebook tour so cross-references still line up.


---
# 1. Introduction

## 1.1 What is a ratio study?

A **ratio study** compares what a jurisdiction *assesses* a property is worth (the **assessed
value**, AV) against a **market value proxy** — usually a screened, time-adjusted sale
price (SP).

The basic unit of analysis is the **assessment-to-sale ratio (ASR)**:

$$\mathrm{ASR} = \frac{\text{assessed value}}{\text{time-adjusted sale price}}$$

Note that:

- **Market value cannot be observed directly.** A single sale is *not* market value; it is
  one observation of it. Ratio study statistics therefore describe a *population* of sales, and
  **cannot be used to judge the valuation of an individual property**.
- **Validity rests on representativeness.** The sample must be representative of the
  population of properties being assessed, and sold and unsold properties must be valued the same way. Where they are
  not — *sales chasing* — the study reports uniformity that does not exist.

From the *distribution* of ASRs across a sample, a ratio study answers three questions:

| Question | Concept | Indicators |
|---|---|---|
| Are we assessing at the right **level**? | Level of valuation | Median, Mean, Weighted Mean |
| Are similar properties treated **alike**? | Uniformity / horizontal equity | COD, COV, Coefficient of Concentration |
| Are lower value and higher value properties treated **alike**? | Vertical equity | **VEI**, PRB, PRD |


## 1.2 Why it matters, and to whom

The primary users are those creating, using or
interpreting ratio studies for mass valuation:

- **Assessment jurisdictions** must demonstrate compliance. A ratio study is the evidence.
- **Valuers and modellers** use it as the acceptance test on a CAMA model, and as a
  diagnostic tool that points *where* the model is deficient.
- **Oversight agencies** use it to monitor, equalize (directly or indirectly), and order
  revaluations.
- **Taxpayers, appeal boards and advocacy groups** use it to detect systematic inequity.
  Note that while a ratio study cannot judge one property, its statistics *can* be used to
  adjust an appealed assessment to the common level.
- **Researchers, journalists, and mortgage-portfolio analysts**.


## 1.3 Motivation for this open-source package

Ratio studies have been run for decades, mostly in spreadsheets, bespoke SAS/SPSS scripts,
or proprietary CAMA modules. 

That has three costs: results are hard to reproduce, newer
literature is slow to reach practice, and every jurisdiction rebuilds the same machinery.

`ratio_study` was built to fix that:

- **IAAO Standard-complete.** The ratio indicators, their acceptable ranges, the
  confidence-interval formulas, the outlier rules and the new VEI.
- **Beyond the Standard.** Additional vertical equity tests (Quintos' Gini /
  Modified Kakwani, Kruskal-Wallis), plus a moving-median measure and ML/SHAP inequity
  diagnostics, in the same API.
- **Reproducible.** One call, seeded, versioned, scriptable in CI. No spreadsheet
  archaeology — a report should let an interested party *replicate* the results.
- **Report-ready.** PDF, Excel, CSV and PNG with automated commentary written for a
  non-technical reader.
- **Backend-agnostic.** pandas, polars or PyArrow in, same backend out, via
  [narwhals](https://github.com/narwhals-dev/narwhals).
- **Open.** GPL-3.0. Jurisdictions should be able to read, audit and improve the code that
  measures the fairness of their own assessments.

---
# 2. Basic ratios

## 2.1 The project page

Source, issue tracker, changelog and the full API documentation live at the project
repository:

> **https://github.com/mpac-asma/ratio_study**

Principal references the package implements:

- IAAO. 2026. *Standard on Ratio Studies* (October 2026). Kansas City
- Gloudemans, R. & Almy, R. 2011. *Fundamentals of Mass Appraisal.* IAAO
- Bonett, D.G. & Seier, E. 2006. "Confidence Intervals for a Coefficient of Dispersion in
  Nonnormal Distributions." *Biometrical Journal* 48(1)
- Clapp, J.M. 1989. "Sample Size in Ratio Studies." *Property Tax Journal* 8(3)
- Quintos, C. 2020, 2021. Gini measure / Gini decomposition. *JPTAA* 17(2), 18(2)
- Efron, B. & Tibshirani, R.J. 1993. *An Introduction to the Bootstrap.*


## 2.2 Load the package

The core install is deliberately light:

```
pip install ratio_study
```

Optional extras switch on the heavier features
used later in this notebook:

| Extra | Enables | Section |
|---|---|---|
| `ratio_study[spatial]` | ratio heat maps, Moran's I | 3 |
| `ratio_study[ml]` | ASR predictability / SHAP diagnostics | 5 |
| `ratio_study[interactive]` | Plotly vertical-equity charts | 6 |

Install everything this notebook needs with:

```
pip install "ratio_study[spatial,ml,interactive]"
```

In [ ]:
# Uncomment on first run.
# %pip install "ratio_study[spatial,ml,interactive]"

In [ ]:
import ratio_study as rs
import pandas as pd
import numpy as np
from pathlib import Path

# inline plotting must be enabled AFTER importing ratio_study
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

sample_df = rs.load_sample_dataset()

MARKET_VALUE = 'time_adjusted_sale_price'   # the market value proxy
ASSESSMENT   = 'assessment_value_new'       # the value under test (current CAMA model)
PRIOR_MODEL  = 'assessment_value_old'       # a previous model, for comparisons

results_directory = Path().resolve()   # outputs are written here
print('results will be written to:', results_directory)

print('ratio_study version:', rs.__version__)
print(f'{len(sample_df):,} sales, {sample_df.shape[1]} columns')

## 2.3 Load a dataset

The package ships with an anonymised residential sales file so every example below is
reproducible without supplying your own data. 

`load_sample_dataset()` returns a pandas dataframe by
default; pass `'polars'` or `'pyarrow'` to get the same data in another backend.

In [ ]:
sample_df = rs.load_sample_dataset()

print(f'{len(sample_df):,} sales, {sample_df.shape[1]} columns')
sample_df.head()

Two columns carry the study. Name them explicitly rather than relying on argument
position.

In [ ]:
MARKET_VALUE = 'time_adjusted_sale_price'   # the market value proxy
ASSESSMENT   = 'assessment_value_new'       # the value under test (current CAMA model)
PRIOR_MODEL  = 'assessment_value_old'       # a previous model, for the comparison in 2.10 of the basics notebook

sample_df[[MARKET_VALUE, ASSESSMENT, PRIOR_MODEL]].describe().round(0)

Note the denominator is the **time-adjusted** sale price. 

Adjusting sales for time allows one to use sales over multiple years.

## 2.4 Generate basic ratio indicators

One call. `ratios()` computes the indicators, formats them for presentation, and attaches
confidence intervals to the headline measures by default — which the Standard requires:
*every* ratio study should include measures of level, of uniformity, **and** confidence
intervals showing statistical reliability.

In [ ]:
br = rs.ratios(sample_df,
          assessment_header=ASSESSMENT,
          market_value_header=MARKET_VALUE)
br

### Add stratifications

### Reading the columns

`Level` / `Group` name the stratum, `Count` is the sample size for that row, while the prefix `_ci` indicates a confidence interval.

### Choosing the indicator set

The default is `['Median', 'Mean', 'Weighted_Mean', 'CoD', 'PRD', 'PRB', 'VEI']`. `VEM` is **opt-in**: it is the slowest indicator and has no IAAO standard, so the three `Inequity_*` columns appear only when `'VEM'` is in `indicators` (see section 6.4).

Ask for more by passing the argument 'indicators' with a list of the required indicators.

| Indicator | Family |
|---|---|
| `Median`, `Mean`, `Weighted_Mean` | Level of valuation |
| `CoD` | Uniformity — the recommended measure |
| `CoD_rsd`, `CoV`, `CoC_15`, `FSD` | Other uniformity measures |
| `VEI` | Vertical equity — **the preferred measure** |
| `PRB` | Vertical equity |
| `PRD` | Vertical equity — **being retired** |
| `VEM` | Vertical equity — moving-median measure (not part of the IAAO Standard; opt-in) |
| `MKI`, `RRI` | Vertical equity (Gini) |
| `RMSE`, `MAE`, `MAPE` | Accuracy |

In [ ]:
ALL_INDICATORS = ['Median', 'Mean', 'Weighted_Mean',
                  'CoD', 'CoD_rsd', 'CoV', 'CoC_15', 'FSD',
                  'VEI', 'PRB', 'PRD', 'VEM', 'MKI', 'RRI',
                  'RMSE', 'MAE', 'MAPE']

full_study = rs.ratios(sample_df,
                       assessment_header=ASSESSMENT,
                       market_value_header=MARKET_VALUE,
                       indicators=ALL_INDICATORS)

full_study         # transposed - one indicator per row, easier to read

## 2.5 Simplified standard pass checks

`descriptive=True` grades the study against standards and writes a plain-language
comment — **only where an indicator falls short**.

In [ ]:
rs.ratios(sample_df,
          assessment_header=ASSESSMENT,
          market_value_header=MARKET_VALUE,
          descriptive=True,
          indicators=['Median', 'CoD', 'PRB'],
          group_by='property_type')

The comments are gated on the confidence intervals (when available), not just the point estimate.

### Choosing the right standards for your property class

Ratio studies work to the range
for their property class and market profile:

| Property category | Profile | COD range |
|---|---|---|
| Residential improved | general | 5.0 – 15.0 |
| | newer / homogeneous, condominiums | 4.0 – 10.0 |
| | rural, seasonal, manufactured homes | 10.0 – 20.0 |
| Multi-unit apartments | general | 7.0 – 20.0 |
| Commercial / industrial / retail | general | 10.0 – 25.0 |
| Vacant land | general | 10.0 – 20.0 |
| Agricultural land | | 10.0 – 25.0 |

The Standard widens these substantially for **blighted, distressed and emerging** market
areas (e.g. residential 10.0–25.0 distressed, 10.0–35.0 emerging).

Level of valuation should be **0.90 to 1.10** for any class, with a target of 1.00.

In [ ]:
# Residential improved, general profile, per the Standard.
# CoD is expressed as a fraction here, so 5.0-15.0 becomes 0.05-0.15.
# set a dictionary with the standards, and pass to ratios()
assessment_standards = {'Median': (0.90, 1.10),     # level of valuation
                        'CoD':    (0.05, 0.15),     # residential improved
                        'VEI':    (-10.0, 10.0),    # preferred measure
                        'PRB':    (-0.05, 0.05),
                        'PRD':    (0.98, 1.03)}     # retiring

rs.ratios(sample_df,
          assessment_header=ASSESSMENT,
          market_value_header=MARKET_VALUE,
          indicators=['Median', 'CoD', 'VEI', 'PRB', 'PRD'],
          descriptive=True,
          assessment_standards=assessment_standards)

### Checking each stratum against the overall level

With `descriptive=True` and a `group_by`, each stratum is also compared with the overall level of assessment (IAAO 11.2.1: within 5 percent) and with its peers (IAAO 11.2.2: not probably differing by more than 10 percent).

`level_of_assessment` picks the statistic used: `'Median'`, `'Mean'` or `'Weighted_Mean'` (default: the Median when computed, else the Weighted Mean, else the Mean). `strata_uniformity_checks=False` turns both checks off. The 11.2.2 comparison uses confidence intervals, so it needs per-group intervals (`bootstrap_args={'groups': True}`); without them only the 11.2.1 check can run.

In [ ]:
rs.ratios(sample_df,
          assessment_header=ASSESSMENT,
          market_value_header=MARKET_VALUE,
          group_by='property_type',
          indicators=['Median', 'Mean', 'CoD'],
          descriptive=True,
          level_of_assessment='Mean')

### Confidence intervals

Overall 95% BCa bootstrap intervals for `Median`, `CoD`, `PRD` and `PRB` are on by
default. Everything about them is adjustable — which indicators, what level, how many
resamplings, and the seed that makes a published number reproducible.

`ci_indicators` also brings its indicators into the study: naming one that is not in `indicators` adds it, rather than dropping the interval. Bootstrap intervals are available for `MKI`, `RRI`, `VEI` and `VEM` as well (`MKI_ci`, `RRI_ci`, `VEI_ci`, `Inequity_severity_ci`, `Inequity_prevalence_ci`).

The Standard asks for **90% two-tailed** (or 95% one-tailed) before concluding noncompliance.

In [ ]:
rs.ratios(sample_df,
          assessment_header=ASSESSMENT,
          market_value_header=MARKET_VALUE,
          descriptive=True,
          ci_indicators=['Median', 'CoD', 'PRD'],
          ci_conf_level=90,
          ci_seed=7,
          bootstrap_args={'resamplings': 500})

There is a second family of intervals: the **closed-form formulas from the Standard**.

| Statistic | Method |
|---|---|
| Mean | Student's *t* about the mean |
| Median | Rank formula, `j = z·√n / 2` |
| Weighted mean | Ratio-estimator standard error |
| Median, n ≤ 30 | Clapp (1989) binomial interpolation tables |
| COD | Bonett–Seier |

Switch families with one argument. These carry no minimum-sample gate — the binomial
median tables reach down to 5 sales at 90% and 6 at 95% — which is what makes them the
right choice for thin strata.

In [ ]:
rs.ratios(sample_df,
          assessment_header=ASSESSMENT,
          market_value_header=MARKET_VALUE,
          indicators=['Median', 'CoD', 'PRD', 'PRB'],
          ci_mode='distribution',
          ci_conf_level=95,
          descriptive=True)

### Seeing the settings you did not choose

Most of what governs an interval is a default nobody typed: the level, the method, the
number of resamplings, the size of each resample, the minimum count, the seed. A published
interval can therefore rest on settings that were never reviewed — and the Standard asks a report
to describe *"statistics used, and reliability and proper use of results"*.

`print_ci_settings` (on by default) prints the resolved settings before the study runs, and
marks anything that differs from the package default with `<-- set`, so what you chose is
visually separable from what you inherited. Set it to `False` in loops or CI.

In [ ]:
settings_demo = rs.ratios(sample_df,
                          assessment_header=ASSESSMENT,
                          market_value_header=MARKET_VALUE,
                          indicators=['Median', 'CoD', 'PRD', 'PRB'],
                          ci_conf_level=90,
                          ci_seed=7,
                          bootstrap_args={'resamplings': 500, 'method': 'percentile'}
                          , print_ci_settings=True)
settings_demo

Note the last line of the default summary when no seed is given: *"intervals vary slightly
between runs"*. That is worth acting on before publishing a number — `ci_seed` makes the
interval reproducible, which is what lets a reader replicate the study, as the Standard recommends.

### `bootstrap_resample_pct` — how certain would I be on a smaller sample?

The classical bootstrap draws *n* observations from a sample of *n*, so the interval
estimates the sampling error of the study you actually ran.

`bootstrap_resample_pct` draws fewer — an **m-out-of-n bootstrap**. That answers a different
and very practical question: *what would my confidence interval look like if I had only had
a fraction of these sales?*

That matters because sample size is central to the Standard's guidance. Before a study is designed,
the Standard notes that the required sample size has to be projected from the
dispersion you expect, *"which is not known during the design and planning stage"*. Running
the bootstrap at a reduced draw on last cycle's data is a direct, assumption-free way to
estimate what precision a smaller sample would buy — and whether a thin stratum next year
will produce an interval narrow enough to conclude anything with.

Two details:

- Give it a fraction (`0.2`) or a percentage (`20`); both read the same.
- Each draw is floored at **20 sales** (or *n*, if the sample is smaller than that), so a 20%
  draw from 50 sales takes 20, not 10. Below about 20 the indicators stop meaning anything —
  a COD on 8 sales is noise — and the floor is also the `min_count_threshold` the package
  uses everywhere else.

In [ ]:
precision_curve = []
for pct in (1.00, 0.50, 0.25, 0.10, 0.05, 0.01):
    row = rs.ratios(sample_df,
                    assessment_header=ASSESSMENT,
                    market_value_header=MARKET_VALUE,
                    indicators=['Median', 'CoD'],
                    ci_conf_level=90,
                    ci_seed=7,
                    print_ci_settings=False,
                    bootstrap_args={'resamplings': 400, 'resample_pct': pct}).iloc[0]

    low, high = (float(x) for x in row['Median_ci'].strip('()').split(','))
    precision_curve.append({
        'resample %': f'{pct:.0%}',
        'sales drawn': max(20, round(len(sample_df) * pct)),
        'Median': row['Median'],
        'Median 90% CI': row['Median_ci'],
        'CI width': round(high - low, 4),
        'CoD 90% CI': row['CoD_ci'],
    })

pd.DataFrame(precision_curve).set_index('resample %')

The point estimate never moves — resampling cannot change the sample statistic. What moves
is the **width**, and it widens roughly as $\sqrt{n/m}$: quartering the draw about doubles the
interval.

Read the table as a sample-size planning tool. If a 25% draw still gives a median interval
tight enough to test against the Standard's range of 0.90–1.10, then a stratum a quarter this
size would also support a conclusion. If a 5% draw does not, then thin strata next cycle
will need the remedies the Standard suggests — a longer sales period, combined strata, or independent
value estimates.

The floor is visible at the bottom of the table: 1% of 12,583 is 126, comfortably above 20,
but on a 50-sale stratum the same 1% request would draw 20.

In [ ]:
# On a small stratum the floor binds: 20% and 40% both draw 20 sales,
# so with one seed they produce exactly the same interval.
small = sample_df.head(50)
for pct in (0.20, 0.40):
    ci = rs.ratios(small, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                   indicators=['Median'], ci_conf_level=90, ci_seed=3, print_ci_settings=False,
                   bootstrap_args={'resamplings': 300, 'resample_pct': pct}).iloc[0]['Median_ci']
    print(f'n=50, resample_pct={pct:.0%} -> draw {max(20, round(50 * pct)):>2} sales, '
          f'Median 90% CI {ci}')

## 2.6 Horizontal Equity Analysis - stratification

An overall study that passes can still hide a stratum that badly fails. `group_by`
runs the whole study again within each stratum and stacks the results under the overall
row. This is what the Standard calls stratification.

Two standards apply *between* strata:

- each stratum's level within **5 percent** — not 5 percentage points — of the overall level
- classes not provably differing from each other by more than **10 percent**

In [ ]:
rs.ratios(sample_df,
          assessment_header=ASSESSMENT,
          market_value_header=MARKET_VALUE,
          group_by='property_type',
          descriptive=True)

Pass a list to get several independent groupings in one table, and a **nested list** to get
the cross-tabulation. The Standard recommends stratifying on more than one characteristic
simultaneously when evaluating valuation quality. Sales with a missing group value are not dropped: they form a `Missing` group, listed last.

In [ ]:
nested = rs.ratios(sample_df,
                   assessment_header=ASSESSMENT,
                   market_value_header=MARKET_VALUE,
                   group_by=['quality_class',
                             'property_type',
                             ['quality_class', 'property_type']],
                   descriptive=True)

nested

Group-level confidence intervals are off by default because they are the expensive (slow) part.
`confidence_intervals_aggs='both'` turns them on alongside the overall ones. Groups below
`min_count_threshold` (20 sales) are left blank rather than given an interval built on too
little evidence — the Standard warns against concluding anything from an
unreliably small stratum.

In [ ]:
rs.ratios(sample_df,
          assessment_header=ASSESSMENT,
          market_value_header=MARKET_VALUE,
          group_by='property_type',
          descriptive=True,
          confidence_intervals_aggs='both',
          ci_indicators=['Median', 'CoD'],
          ci_seed=7)

## 2.7 The IQR outlier filter

Ratio studies are sensitive to a handful of bad sales: one extreme ratio can move a mean,
a PRD or a weighted mean on its own. The Standard sets out the procedure in
three steps, and the order matters:

1. **Identify** candidate outliers.
2. **Scrutinise** them — correct data errors, and re-screen non-market sales.
3. **Trim** only what would otherwise compromise the representativeness of the statistics.

> *"Outliers should never be removed simply to generate better ratio
> statistics, which serves only to destroy the representativeness and validity of the
> study."*

The package implements the Standard's interquartile-range rule. Four arguments control
it, and one more controls what you are told about it:

| Argument | Default | What it does |
|---|---|---|
| `iqr_trim_multiple` | `0` (off) | IQR multiplier. `1.5` and `3.0` are the two the Standard names |
| `iqr_log_transform` | `True` | Compute the bounds on `log(ASR)` rather than the raw ratio |
| `iqr_min_cod` | `0.06` | Skip the trim on samples that have CoDs lower than this |
| `iqr_group_by` | `None` | Compute the bounds independently within strata |
| `iqr_filter_result` | `'print'` | Report — and optionally export — what was removed |

### Choosing the multiple

The Standard names two: **1.5 × IQR**, which typically flags around 5% of ratios, and
**3.0 × IQR**, which flags around 1% — the most extreme only. Which is right depends on how
well the sales were screened before they reached you:

- **Thoroughly verified sales** → prefer `3`. The obvious errors are already gone, so a
  wide net mostly removes legitimate market variation.
- **Lightly screened sales** → `1.5` is defensible, because more of what it catches really
  is bad data.

The Standard then caps the result regardless of the multiple: **no more than 5–10%** of a
large sample, and no more than 10% (20% in the most extreme cases) for samples under 30.
Trimming must be a **single pass** — never repeated rounds.

Let's see what each multiple actually removes here.

In [ ]:
for multiple in (1.5, 3.0):
    print(f'--- iqr_trim_multiple = {multiple} ---')
    rs.ratios(sample_df,
              assessment_header=ASSESSMENT,
              market_value_header=MARKET_VALUE,
              iqr_trim_multiple=multiple,
              print_ci_settings=False)
    print()

Both land far under the Standard's cap, which is what you would hope for on a
pre-screened file. The summary printed above is the `iqr_filter_result='print'` default —
more on that in a moment.

### Why the bounds are computed on the log

`iqr_log_transform=True` is the default, and it matches the Standard directly: the ASR
distribution is skewed right, so quartiles taken on the raw scale push the upper bound out
and pull the lower bound in. The result is that raw-scale trimming **catches low outliers
and misses high ones**. The Standard puts it plainly: a log transform *"tends to achieve greater
symmetry and identify fewer high and more low ratios as outliers."*

The asymmetry is easy to see — the ratio is bounded below at 0 but unbounded above, so
"half the value" and "twice the value" are the same distance in log space and very
different distances on the raw scale.

In [ ]:
comparison = []
for log_transform in (True, False):
    result = rs.ratios(sample_df,
                       assessment_header=ASSESSMENT,
                       market_value_header=MARKET_VALUE,
                       iqr_trim_multiple=3,
                       iqr_log_transform=log_transform,
                       iqr_filter_result=None,        # quiet - we tabulate below
                       return_dict=True)
    dropped = result['IQR_trimmed_sales']
    median_asr = (sample_df[ASSESSMENT] / sample_df[MARKET_VALUE]).median()
    comparison.append({
        'log_transform': log_transform,
        'trimmed': len(dropped),
        'below median ASR': int((dropped['ASR'] < median_asr).sum()),
        'above median ASR': int((dropped['ASR'] > median_asr).sum()),
        'lowest ASR kept out': round(dropped['ASR'].min(), 4),
        'highest ASR kept out': round(dropped['ASR'].max(), 4),
    })

pd.DataFrame(comparison).set_index('log_transform')

### The `min_cod` gate

`iqr_min_cod=0.06` skips the trim entirely on a sample whose COD is already below 6%.

This exists because of a warning in the Standard: in a tight, homogeneous sample the IQR
is small, so the rule flags *a large share of the data* as "outliers" even though the ratio
differences between them are trivial. Trimming there removes representative sales and
inflates the apparent quality of the study — the same distortion the Standard warns about when it
says a very low COD should be treated as a diagnostic warning, not a good result.

The gate bites hardest at the tighter multiple, so this example uses `1.5`. Take one
homogeneous stratum — a single quality class within one neighbourhood — and turn the gate
off with `iqr_min_cod=0`.

In [ ]:
# A deliberately tight stratum.
tight = sample_df[(sample_df['neighbourhood'] == 29) & (sample_df['quality_class'] == 5)]
tight_cod = rs.ratios(tight, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                      indicators=['Median', 'CoD'],
                      return_dict=True, print_ci_settings=False)['df_all_ratios']['CoD'].iloc[0]
print(f'{len(tight)} sales, COD = {tight_cod:.2%}  '
      f'-- already inside the 5.0-15.0 residential range of the Standard\n')

for gate in (0.06, 0.0):
    print(f'--- iqr_min_cod = {gate} ---')
    rs.ratios(tight, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
              iqr_trim_multiple=1.5, iqr_min_cod=gate, print_ci_settings=False)
    print()

With the gate on, nothing is touched. With it off, the 1.5 × IQR rule strips several sales
out of an already-uniform stratum — a chunk of a sample that had no dispersion problem to
begin with, and close to the Standard's ceiling. Those sales are not outliers in any
meaningful sense; they are just the tails of a tight distribution.

Set `iqr_min_cod=0` only when you have a specific reason to trim regardless of dispersion.

### Trimming within strata

Without `iqr_group_by`, the bounds come from the whole sample: one set of thresholds, applied
everywhere. That is the right default when the jurisdiction is reasonably homogeneous.

With `iqr_group_by`, the bounds are computed **independently inside each stratum**, so a
neighbourhood that is legitimately expensive — or legitimately more variable — is judged
against its own peers rather than against the jurisdiction. A sale is trimmed if it is an
outlier under **any** of the groupings you pass, and the `min_cod` gate is applied per group,
so tight strata are individually skipped.

This matters because the Standard pushes the analyst toward stratification: an outlier
relative to the whole file and an outlier relative to its own neighbourhood are different
claims.

In [ ]:
print('--- pooled: one set of bounds for the whole file ---')
pooled = rs.ratios(sample_df, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                   iqr_trim_multiple=3, return_dict=True, print_ci_settings=False)

print('\n--- within neighbourhood ---')
by_nbhd = rs.ratios(sample_df, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                    iqr_trim_multiple=3, iqr_group_by='neighbourhood', return_dict=True, print_ci_settings=False)

print('\n--- within neighbourhood AND within property type (union of both) ---')
either = rs.ratios(sample_df, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                   iqr_trim_multiple=3, iqr_group_by=['neighbourhood', 'property_type'],
                   return_dict=True, print_ci_settings=False)

print('\n--- within the neighbourhood x property type cell (nested) ---')
nested_trim = rs.ratios(sample_df, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                        iqr_trim_multiple=3, iqr_group_by=[['neighbourhood', 'property_type']],
                        return_dict=True, print_ci_settings=False)

The four runs disagree about which sales are extreme, which is the point — an outlier is
only ever an outlier *relative to a reference group*.

In [ ]:
sets = {'pooled': pooled, 'by neighbourhood': by_nbhd,
        'either grouping': either, 'nested cell': nested_trim}
overlap = pd.DataFrame({
    'trimmed': {k: len(v['IQR_trimmed_sales']) for k, v in sets.items()},
    'also trimmed when pooled': {
        k: len(v['IQR_trimmed_sales'].index.intersection(pooled['IQR_trimmed_sales'].index))
        for k, v in sets.items()},
})
overlap['unique to this grouping'] = overlap['trimmed'] - overlap['also trimmed when pooled']
overlap

### `iqr_filter_result` — reporting what was removed

The Standard is unambiguous that a trim has to be reported, not just performed:

> *"Statistics calculated from trimmed distributions are not directly comparable to
> statistics from untrimmed distributions... Ratio study reports or accompanying
> documentation should clearly state the basis for excluding outlier ratios."*

and it adds that *"if a trimming method has been used to reject ratios from the
sample, this fact must be stated in ratio study reporting, and outliers removed should be
identified."*

So the default is to say something. `iqr_filter_result` controls how much:

| Value | Behaviour |
|---|---|
| `'print'` *(default)* | Print the count, the share of the sample, and the total assessed and market value removed |
| `'save'` | Also write the removed records to a `.csv.gz` in `export_directory`, then print the path and the summary |
| a directory | As `'save'`, written there |
| a file path | Written exactly there, forced to `.csv.gz` |
| `None` or `''` | Silent — for loops and CI, as used in the comparison cells above |

The summary alone is enough for the report narrative the Standard asks for. The export is what
lets someone *check* the exclusions.

In [ ]:
# The default: a summary, no file.
_ = rs.ratios(sample_df,
              assessment_header=ASSESSMENT,
              market_value_header=MARKET_VALUE,
              iqr_trim_multiple=3,
              iqr_group_by='neighbourhood',
              iqr_filter_result='print', print_ci_settings=False)

In [ ]:
# 'save' writes the removed records next to the other outputs and says where.
audited = rs.ratios(sample_df,
                    assessment_header=ASSESSMENT,
                    market_value_header=MARKET_VALUE,
                    iqr_trim_multiple=3,
                    iqr_group_by='neighbourhood',
                    iqr_filter_result='save',
                    export_directory=results_directory,
                    model_name='DemoModel',
                    return_dict=True, print_ci_settings=False)

audited['IQR_trimmed_sales_export']

The export carries the **whole sales record**, not just the two columns the study needed —
the Standard asks for the flagged sales to be scrutinised, and you cannot scrutinise a ratio
without the property behind it. The removed records are also always available in memory
under `IQR_trimmed_sales`, whatever `iqr_filter_result` is set to.

In [ ]:
dropped = audited['IQR_trimmed_sales']
print(f'{len(dropped)} records, {dropped.shape[1]} columns\n')

dropped[[MARKET_VALUE, ASSESSMENT, 'ASR', 'property_type',
         'neighbourhood', 'living_area', 'quality_class']].sort_values('ASR').head(10)

Reading that table is the scrutiny step the Standard asks for. Vacant land with a living area of 0, or a
rooming house priced like a detached home, are **data or classification problems** to fix at
source — not sales to quietly delete. Anything you cannot explain should be verified before
it is trimmed.

`ratios_report()` takes the same argument, so a scheduled report can drop an audit file
beside its PDF.

In [ ]:
iqr_report = rs.ratios_report(sample_df,
                              assessment_header=ASSESSMENT,
                              market_value_header=MARKET_VALUE,
                              group_by='property_type',
                              descriptive=True,
                              iqr_trim_multiple=3,
                              iqr_group_by='neighbourhood',
                              iqr_filter_result='save',
                              model_name='DemoModel_Trimmed',
                              export_directory=results_directory,
                              export_in_compacted_file='',
                              return_dict=False, print_ci_settings=False)

sorted(p.name for p in results_directory.glob('*DemoModelTrimmed*'))

The PDF also picks this up on its own: when a trim is active, `ratios_report()` prepends a
note to the opening comments recording how many sales were removed and on what rule —
the methodology disclosure the Standard asks for, written for you.

### Was the trim worth it?

Finally, the honest check. `compare_ratios()` puts the trimmed and untrimmed studies side by
side, which is the comparison the Standard effectively demands before you publish trimmed
numbers.

In [ ]:
untrimmed = rs.ratios(sample_df, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                      indicators=['Median', 'Mean', 'Weighted_Mean', 'CoD', 'VEI', 'PRB', 'PRD'],
                      iqr_filter_result=None, return_dict=True, print_ci_settings=False)
trimmed_study = rs.ratios(sample_df, assessment_header=ASSESSMENT, market_value_header=MARKET_VALUE,
                          indicators=['Median', 'Mean', 'Weighted_Mean', 'CoD', 'VEI', 'PRB', 'PRD'],
                          iqr_trim_multiple=3, iqr_group_by='neighbourhood',
                          iqr_filter_result=None, return_dict=True, print_ci_settings=False)

rs.compare_ratios(untrimmed, trimmed_study,
                  suffixes=('_untrimmed', '_trimmed'),
                  difference=True, pct_chg=True)

Note which indicators move. The **median** barely shifts — it is resistant by construction,
which is why the Standard prefers it. The **mean**, **weighted mean** and **PRD** move more, because
they are the ones extreme ratios actually distort. If trimming changes your headline
conclusion, that is a finding about the fragility of the conclusion, not a licence to keep
trimming.

## 2.8 Visualisations — graphs

The Standard lists the visualisations useful in a ratio study — tables, bar charts, histograms,
scatterplots and maps — and what they are for: showing sample representativeness,
non-normality, level, uniformity, value-related bias, and outliers.

It also carries a warning worth repeating: **graphics alone should not be used to draw
formal conclusions.** They shed light; the hypothesis tests decide.

Every chart follows the same three-way convention:

| Call | Returns | Use for |
|---|---|---|
| `f(df, ...)` | a matplotlib `Figure` | inline display in this notebook |
| `f(df, ..., graph_filepath='chart.png')` | the saved path | PNG export |
| `f(df, ..., return_object=True)` | a report-ready flowable | embedding in the PDF |

and the same first three arguments — the sales frame, the assessment column and the market
value column. The ratio is derived for you; pass `asr_col=` instead if your frame already
has one.

### Scatterplot — the workhorse

The ratio against any variable, with a lowess trend and the compliance band marked. A flat
cloud is a model behaving; a visible slope is the ratio moving with whatever is on the x
axis. Against sale price this is the graphical companion to the VEI and PRB.

In [ ]:
rs.asr_scatter(sample_df,
               assessment_header=ASSESSMENT,
               market_value_header=MARKET_VALUE,
               x=MARKET_VALUE,
               log_x=True,
               x_label='Time-adjusted sale price (log scale)',
               title='Ratio against sale price')

**Zooming.** `zoom=True` (the default) trims both axes to where the points actually are —
the 1st–99th percentile of the x variable, and 95% coverage of the ratios inside that
window — the same treatment the VEM chart uses. Without it, a handful of very large sales
and one ratio of 6 take most of the plot area and squeeze the cloud that carries the
finding into a corner. The median line is always kept in view, so zooming can never hide
the statistic the chart is about.

Note also that these charts no longer draw the 0.90–1.10 acceptance band. On a scatter of
individual sales the band invites exactly the reading the Standard warns against — judging single
properties against a standard that describes a population. The median line stays.

In [ ]:
rs.asr_scatter(sample_df,
               assessment_header=ASSESSMENT,
               market_value_header=MARKET_VALUE,
               x=MARKET_VALUE,
               log_x=True,
               zoom=False,                      # the full range, tails and all
               x_label='Time-adjusted sale price (log scale)',
               title='Ratio against sale price - zoom OFF')

The same data with the default zoom is the chart above: same cloud, but readable. The
marginal histograms also carry more buckets now (`marginal_bins=120`), so the shape of both
distributions is visible rather than blocky.

The same chart against a property characteristic answers a different question: is the model
mis-pricing a *feature* rather than a price level? The Standard asks for exactly this check —
consistency across size groups, age groups and geographic areas — **before** a formal
vertical equity analysis.

In [ ]:
rs.asr_scatter(sample_df,
               assessment_header=ASSESSMENT,
               market_value_header=MARKET_VALUE,
               x='living_area',
               x_label='Living area (sq ft)',
               title='Ratio against living area')

### Histogram — the shape of the distribution

The Standard asks the analyst to examine the distribution *first*: performance measures built on
the mean or standard deviation mislead when the sample is not normal, and ratio study
samples typically are not. A bimodal histogram is the fastest way to see that two
populations have been mixed.

In [ ]:
rs.asr_histogram(sample_df,
                 assessment_header=ASSESSMENT,
                 market_value_header=MARKET_VALUE,
                 bins=60,
                 title='Distribution of assessment-to-sale ratios')

### Boxplot by stratum — horizontal equity at a glance

The visual form of a grouped ratio table, and the chart type the Standard illustrates. Boxes at different **heights** are differing levels of valuation between strata
— the Standard's five-percent test. Boxes of different **sizes** are differing uniformity —
the COD story.

In [ ]:
rs.asr_boxplot(sample_df,
               assessment_header=ASSESSMENT,
               market_value_header=MARKET_VALUE,
               by='property_type',
               min_count=30,
               order_by='median',
               title='Ratio distribution by property type')

`show_values=True` prints each group's median ratio above its box. The box shows the shape;
the number is what goes into the report — and the test a reader is actually making
is each stratum's level against the overall one, which is far easier to do with the figures
on the chart.

In [ ]:
rs.asr_boxplot(sample_df,
               assessment_header=ASSESSMENT,
               market_value_header=MARKET_VALUE,
               by='property_type',
               min_count=30,
               order_by='median',
               show_values=True,                # value_fmt='{:.1%}' for percentages
               title='Ratio distribution by property type, with group medians')

### Facet grid — small multiples

An overall scatter can look flat while individual neighbourhoods each slope hard in
opposite directions. Faceting is how that shows up: the same axes repeated across strata,
compared by eye against a shared reference line.

In [ ]:
rs.asr_facet(sample_df,
             assessment_header=ASSESSMENT,
             market_value_header=MARKET_VALUE,
             x=MARKET_VALUE,
             col='quality_class',
             col_wrap=3,
             max_facets=9,
             log_x=True,
             x_label='Time-adjusted sale price (log scale)',
             title='Ratio against price, by quality class')

`row=` adds a second faceting dimension, for a true cross-tabulated grid.

In [ ]:
rs.asr_facet(sample_df,
             assessment_header=ASSESSMENT,
             market_value_header=MARKET_VALUE,
             x='living_area',
             col='quality_class',
             row='condition',
             max_facets=12,
             x_label='Living area (sq ft)',
             title='Ratio against living area, by quality and condition')

`asr_facet` takes the same `zoom` argument, and applies one trimmed range across **all**
panels — a small-multiples grid is only useful if the panels are comparable, so the limits
are measured on the whole frame rather than per panel. Set `zoom=False` to see the full
range in each.

In [ ]:
rs.asr_facet(sample_df,
             assessment_header=ASSESSMENT,
             market_value_header=MARKET_VALUE,
             x=MARKET_VALUE,
             col='quality_class',
             col_wrap=3,
             max_facets=6,
             log_x=True,
             zoom=False,                        # compare with the zoomed grid above
             x_label='Time-adjusted sale price (log scale)',
             title='Ratio against price by quality class - zoom OFF')

### The report's own chart family

`ratios()` can also build the charts destined for the PDF. These come back as report
flowables under `overall_graphs` rather than as inline figures.

In [ ]:
charted = rs.ratios(sample_df,
                    assessment_header=ASSESSMENT,
                    market_value_header=MARKET_VALUE,
                    return_dict=True,
                    graphs_args={'pdf_overall_asr_graphs': True,
                                 'pdf_asr_graphs_x_vars_and_aliases': {
                                     'overall': {'living_area':   'Living area (sq ft)',
                                                 'quality_class': 'Quality class',
                                                 'sale_month':    'Month of sale'},
                                     'overall_histogram': True,
                                     'groups': [], 'groups_mosaic': []}})

print('result keys :', list(charted.keys()))
print('charts built:', len(charted['overall_graphs']))

## 2.9 Visualisations — tables

`return_dict=True` gives the study as data rather than as a presentation object.
`df_all_ratios` is the raw numeric frame; `df_all_ratios_formatted` is the display version,
in which anything not computed is a **blank string rather than `NaN`** — a blank reads as
"not computed here", where `NaN` reads as a result.

In [ ]:
by_quality_class = rs.ratios(sample_df,
                    assessment_header=ASSESSMENT,
                    market_value_header=MARKET_VALUE,
                    group_by='quality_class',
                    indicators=['Median', 'CoD', 'VEI', 'PRB'],
                    return_dict=True)

by_quality_class['df_all_ratios']

In [ ]:
by_quality_class['df_all_ratios_formatted']

## 2.10 The ratios comparison tool

The question a modeller actually asks is *"did my change help?"*. `compare_ratios()` takes
two studies and lays them side by side with the difference and the percentage change.
The Standard recommends exactly this when a study is one in a series.

First, the obvious use: the previous model versus the current one, on the same sales.

In [ ]:
old_model = rs.ratios(sample_df, assessment_header=PRIOR_MODEL,
                      market_value_header=MARKET_VALUE, return_dict=True, print_ci_settings=False)
new_model = rs.ratios(sample_df, assessment_header=ASSESSMENT,
                      market_value_header=MARKET_VALUE, return_dict=True, print_ci_settings=False)

rs.compare_ratios(old_model, new_model,
                  suffixes=('_old', '_new'),
                  difference=True, pct_chg=True)

It also works across different sales counts, so it can answer *"what did the outlier trim
actually do?"* — the before/after comparison the Standard effectively asks for.

In [ ]:
before_trim = rs.ratios(sample_df, assessment_header=ASSESSMENT,
                        market_value_header=MARKET_VALUE, return_dict=True, print_ci_settings=False)
after_trim  = rs.ratios(sample_df, assessment_header=ASSESSMENT,
                        market_value_header=MARKET_VALUE, return_dict=True, print_ci_settings=False,
                        iqr_trim_multiple=3)

rs.compare_ratios(before_trim, after_trim,
                  suffixes=('_no_filter', '_iqr3'),
                  difference=True, pct_chg=True)

And it compares grouped studies, so you can find the strata a model change helped — and
the ones it hurt. Narrow the output with `indicators=`.

In [ ]:
grouping = ['quality_class', 'property_type', ['quality_class', 'property_type']]

old_grouped = rs.ratios(sample_df, assessment_header=PRIOR_MODEL,
                        market_value_header=MARKET_VALUE,
                        group_by=grouping, return_dict=True, print_ci_settings=False)
new_grouped = rs.ratios(sample_df, assessment_header=ASSESSMENT,
                        market_value_header=MARKET_VALUE,
                        group_by=grouping, return_dict=True, print_ci_settings=False)

compared_ratios = rs.compare_ratios(old_grouped, new_grouped,
                                    suffixes=('_old', '_new'),
                                    difference=True, pct_chg=True,
                                    indicators=['Median', 'CoD', 'VEI'])
compared_ratios

In [ ]:
rs.compare_ratios(old_grouped, new_grouped,
                                    suffixes=('_old', '_new'),
                                    difference=True, pct_chg=True,
                                    indicators=['Count', 'Median', 'CoD', 'VEI'])

In [ ]:
# Save the comparison for the model review pack.
compared_ratios.to_csv(results_directory / 'compared_ratios.csv', index=False)
print('written:', results_directory / 'compared_ratios.csv')